# 🚀 AshenAI - Remote GPU Backend for Google Colab & Kaggle

This notebook turns your free **Google Colab (T4 GPU)** or **Kaggle (2x T4 GPU)** into a high-performance, private, OpenAI-compatible streaming backend for your **AshenAI** web interface.

### Supported Models (4-bit NF4 Quantization):
- 🧠 **Qwen 2.5 7B Instruct** (`Qwen/Qwen2.5-7B-Instruct`) - *Recommended, super smart & fast*
- 🦙 **Llama 3.1 8B Instruct** (`meta-llama/Meta-Llama-3.1-8B-Instruct`)
- 🔬 **DeepSeek R1 Distill Qwen 7B** (`deepseek-ai/DeepSeek-R1-Distill-Qwen-7B`) - *Deep reasoning & thinking*
- 💎 **Gemma 2 9B IT** (`google/gemma-2-9b-it`)
- ⚡ **Mistral 7B Instruct v0.3** (`mistralai/Mistral-7B-Instruct-v0.3`)

---

In [ ]:
#@title 1. Install Dependencies (Run this first ~ 1 min)
!pip install -q --upgrade pip
!pip install -q torch transformers accelerate bitsandbytes fastapi uvicorn pydantic pyngrok nest_asyncio

# Download Cloudflare Tunnel (100% Free HTTPS Tunnel with no token required)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O ./cloudflared
!chmod +x ./cloudflared

import torch
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔥 GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB VRAM)")
else:
    print("⚠️ Please change your runtime to GPU (Runtime -> Change runtime type -> T4 GPU)")

In [ ]:
#@title 2. Configure Model & Tunnel Settings

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct" #@param ["Qwen/Qwen2.5-7B-Instruct", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "meta-llama/Meta-Llama-3.1-8B-Instruct", "google/gemma-2-9b-it", "mistralai/Mistral-7B-Instruct-v0.3"] {allow-input: true}
LOAD_IN_4BIT = True #@param {type:"boolean"}
HF_TOKEN = "" #@param {type:"string"} # Optional: Needed for Llama-3 or gated models
TUNNEL_TYPE = "Cloudflare" #@param ["Cloudflare", "ngrok"]
NGROK_AUTH_TOKEN = "" #@param {type:"string"} # Required only if using ngrok
PORT = 8000

import os
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print(f"Selected model: {MODEL_NAME}")

In [ ]:
#@title 3. Load Model and Launch AshenAI Server (Streaming Enabled)
import os, sys, time, json, uuid, asyncio, threading, subprocess, re
from typing import List, Dict, Any, Optional, AsyncGenerator
import torch
from fastapi import FastAPI, Request, HTTPException, Depends
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse, JSONResponse
from pydantic import BaseModel
import uvicorn
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TextIteratorStreamer

app = FastAPI(title="AshenAI Remote Backend")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

print(f"📥 Loading tokenizer & weights for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if LOAD_IN_4BIT and torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        low_cpu_mem_usage=True
    )

print(f"✨ Model successfully loaded on GPU!")

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: Optional[str] = None
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.7
    top_p: Optional[float] = 0.9
    max_tokens: Optional[int] = 2048
    stream: Optional[bool] = True

@app.get("/")
@app.get("/health")
async def health():
    return {"status": "online", "model": MODEL_NAME, "cuda": torch.cuda.is_available()}

@app.get("/v1/models")
async def models():
    return {"data": [{"id": MODEL_NAME, "object": "model"}]}

@app.post("/v1/chat/completions")
async def chat_completions(req: ChatCompletionRequest):
    messages_payload = [{"role": m.role, "content": m.content} for m in req.messages]
    try:
        prompt = tokenizer.apply_chat_template(messages_payload, tokenize=False, add_generation_prompt=True)
    except Exception:
        prompt = "\n".join([f"{m.role}: {m.content}" for m in req.messages]) + "\nassistant:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    gen_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=req.max_tokens or 2048,
        temperature=max(0.01, req.temperature or 0.7),
        top_p=req.top_p or 0.9,
        do_sample=True if (req.temperature or 0.7) > 0.05 else False,
        pad_token_id=tokenizer.pad_token_id
    )
    
    thread = threading.Thread(target=model.generate, kwargs=gen_kwargs)
    thread.start()
    
    async def stream_generator():
        # Initial role chunk
        yield f"data: {json.dumps({'id': 'chatcmpl-' + uuid.uuid4().hex[:8], 'object': 'chat.completion.chunk', 'model': MODEL_NAME, 'choices': [{'index': 0, 'delta': {'role': 'assistant'}, 'finish_reason': None}]})}\n\n"
        for chunk in streamer:
            if chunk:
                yield f"data: {json.dumps({'id': 'chatcmpl-' + uuid.uuid4().hex[:8], 'object': 'chat.completion.chunk', 'model': MODEL_NAME, 'choices': [{'index': 0, 'delta': {'content': chunk}, 'finish_reason': None}]})}\n\n"
            await asyncio.sleep(0.001)
        yield f"data: {json.dumps({'id': 'chatcmpl-' + uuid.uuid4().hex[:8], 'object': 'chat.completion.chunk', 'model': MODEL_NAME, 'choices': [{'index': 0, 'delta': {}, 'finish_reason': 'stop'}]})}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(stream_generator(), media_type="text/event-stream")

# Start Tunnel in Background
if TUNNEL_TYPE == "Cloudflare":
    cf_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time.sleep(3)
    tunnel_url = None
    for _ in range(30):
        line = cf_proc.stdout.readline()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            tunnel_url = match.group(0)
            break
    print("\n" + "="*70)
    print(f"🎉 SUCCESS! YOUR ASHENAI BACKEND URL IS: {tunnel_url}")
    print("Copy & Paste this URL into your AshenAI web interface Settings!")
    print("="*70 + "\n")
elif TUNNEL_TYPE == "ngrok":
    from pyngrok import ngrok
    if NGROK_AUTH_TOKEN:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    url = ngrok.connect(PORT).public_url
    print("\n" + "="*70)
    print(f"🎉 NGROK TUNNEL URL: {url}")
    print("="*70 + "\n")

# Run Uvicorn in current notebook thread
import nest_asyncio
nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=PORT)